In [13]:
from google.colab import files
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print(f"File uploaded successfully: {file_name}")

Saving supply_chain_dataset_unclean.csv to supply_chain_dataset_unclean (3).csv
File uploaded successfully: supply_chain_dataset_unclean (3).csv


In [14]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv(file_name)
print(f"Original dataset shape: {df.shape}")

# Replace messy placeholder strings with NaN
placeholder_values = ['-', 'Null', 'nan', 'N/A', 'none', '', '   ', ' - ', 'NULL', 'NaN', '0']
df = df.replace(placeholder_values, np.nan)

Original dataset shape: (12600, 27)


In [ ]:
# Convert All Data Types and Dates
def parse_order_date_key(val):
    if pd.isna(val) or str(val).strip() in ['0', 'nan', 'None']:
        return pd.NaT
    s = str(val).strip()
    if len(s) == 8 and s.isdigit():
        return pd.to_datetime(s, format='%Y%m%d', errors='coerce')
    else:
        return pd.to_datetime(s, errors='coerce')

df['order_date_key'] = df['order_date_key'].apply(parse_order_date_key)

# Convert all other date columns to datetime
date_cols = ['order_date', 'Ship Date', 'Expected Delivery Date', 'delivery_date', 'order_date_key']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        # Fill missing dates with the mode (most common date) to preserve rows
        mode_date = df[col].mode()[0] if not df[col].mode().empty else pd.Timestamp('2023-01-01')
        df[col] = df[col].fillna(mode_date)

# Convert numeric columns and handle invalid negative values
numeric_cols = [
    'unit_price', 'quantity', 'Order Value', 'total_cost',
    'defect_rate_pct', 'stock_on_hand', 'delay_days', 'lead_time_days', 'Reorder Level', 'shipping_cost'
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        if col in ['unit_price', 'quantity', 'Order Value', 'total_cost', 'stock_on_hand', 'lead_time_days', 'shipping_cost']:
            df.loc[df[col] < 0, col] = np.nan
        # Impute missing numeric values with the median to keep all rows
        df[col] = df[col].fillna(df[col].median())

# Handle missing categorical/text columns by filling with 'Unknown'
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')

print(f"Shape after cleaning & imputing (all rows retained): {df.shape}")

Shape after cleaning & imputing (all rows retained): (12600, 27)


In [16]:
# Cap outliers using IQR method (Winsorization) so no rows are deleted
all_numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in all_numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df[col] = np.where(df[col] < lower_bound, lower_bound, df[col])
    df[col] = np.where(df[col] > upper_bound, upper_bound, df[col])

print(f"Final dataset shape (All {len(df)} rows retained, outliers capped): {df.shape}")

# Verify final data types
print("\nFinal Data Types:")
print(df.dtypes)

Final dataset shape (All 12600 rows retained, outliers capped): (12600, 27)

Final Data Types:
order_id                          object
order_date                datetime64[ns]
supplier                          object
supplier_country                  object
category                          object
Product                           object
unit_price                       float64
quantity                         float64
Warehouse                         object
warehouse_city                    object
Warehouse Country                 object
shipping_mode                     object
destination_city                  object
Destination Country               object
Ship Date                 datetime64[ns]
Expected Delivery Date    datetime64[ns]
delivery_date             datetime64[ns]
delay_days                       float64
lead_time_days                   float64
Order Value                      float64
shipping_cost                    float64
total_cost                       float64
def

In [17]:
output_file = 'supply_chain_dataset_full_cleaned.csv'
df.to_csv(output_file, index=False)
print(f"\nCleaned dataset successfully saved as '{output_file}'")

files.download(output_file)


Cleaned dataset successfully saved as 'supply_chain_dataset_full_cleaned.csv'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>